In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import numpy as np
import time
import os
from pathlib import Path
import json
import math
import requests

In [2]:
df = pd.read_csv('anime_df.csv')
df.columns.values[0] = 'usernames'

In [3]:
df = df.set_index(df.columns[0])
df = df.astype(float)

In [4]:
conf_path = Path("../tauri.conf.json")

with open(conf_path, 'r') as f:
    config = json.load(f)

In [5]:
app_data_path = Path(os.getenv('APPDATA') or Path.home() / ".local/share")

# Path objects handle joining naturally with the / operator
watchlist_path = app_data_path / config['identifier'] / 'watchlist.json'
notwatchlist_path = app_data_path / config['identifier'] / 'notwatchlist.json'

print(f"Watchlist path: {watchlist_path}")
print(f"Not atchlist path: {notwatchlist_path}")
print(f"DEBUG: Próbuję czytać z: {os.path.abspath(watchlist_path)}")

Watchlist path: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\watchlist.json
Not atchlist path: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\notwatchlist.json
DEBUG: Próbuję czytać z: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\watchlist.json


In [6]:
user_data = pd.read_json(watchlist_path)
not_wanted = pd.read_json(notwatchlist_path)


# Konwersja na listę słowników
user_data = user_data.to_dict(orient='records')
not_wanted = not_wanted.to_dict(orient='records')
not_wanted_set = {int(elem['mal_id']) for elem in not_wanted if pd.notna(elem['mal_id'])}
print(not_wanted_set)

{47616, 21507, 56835, 50696, 28683, 37902, 38419, 8728, 37402, 20509, 43555, 49703, 40489, 5163, 42030, 38958, 36914, 20021, 39477, 59959, 33338, 38972, 59968, 51781, 57413, 59465, 25161, 2124, 33360, 16468, 36439, 10842, 33372, 30300, 11359, 60012, 41075, 56442, 40064, 53890, 22661, 52357, 137, 36497, 59029, 21659, 31389, 43683, 19109, 22695, 22699, 57519, 33456, 32437, 8888, 52921, 62653, 54976, 62149, 21195, 1238, 44248, 61149, 37087, 16099, 39652, 60645, 27891, 762, 48897, 42754, 36616, 40206, 54042, 33569, 38693, 38699, 35118, 19759, 32051, 34100, 17205, 40761, 63806, 23359, 39239, 36683, 56141, 31056, 26449, 53588, 63830, 41306, 57691, 15197, 55647, 50532, 61293, 33142, 9591, 21879, 40315, 42364, 17277, 41341, 38784, 33155, 40836, 27525, 33156, 908, 12685, 40334, 36239, 37264, 62352, 37781, 59801, 24991, 28063, 61345, 1953, 18851, 31138, 31145, 45999, 2994, 25011, 54196, 56243, 44983, 59833, 12729, 19391, 18881, 37831, 26057, 61393, 32215, 2520, 31706, 987, 51162, 51168, 36833, 4

In [7]:
def get_recommendations(user, df_orig, df_norm, n=5, k=10):
    knn = NearestNeighbors(n_neighbors=k+1, metric='cosine')
    knn.fit(df_norm)
    
    user_idx = df_norm.index.get_loc(user)
    distances, indices = knn.kneighbors(df_norm.iloc[[user_idx]])
    
    similarities = 1 - distances.flatten()
    neighbor_indices = indices.flatten()
    

    similar_users = pd.Series(similarities[1:], index=df_norm.index[neighbor_indices[1:]])
    
    user_ratings = df_orig.loc[user]
    user_mean = user_ratings[user_ratings != 0].mean() if (user_ratings != 0).any() else 0
    

    neighbor_ratings = df_orig.loc[similar_users.index]
    candidate_mask = (neighbor_ratings > 0).any(axis=0) & (user_ratings == 0)
    candidate_anime = neighbor_ratings.columns[candidate_mask]
    
    predictions = {}
    for anime in candidate_anime:
        id_ref, name = anime.split('_', 1)
        if(int(id_ref)) in not_wanted_set:
            continue        
        
        relevant_indices = similar_users.index[neighbor_ratings[anime] > 0]
        
        if len(relevant_indices) > 0:
            weights = similar_users.loc[relevant_indices]
            norm_ratings = df_norm.loc[relevant_indices, anime]
            
            if weights.sum() > 0: pass
            pred_deviation = np.average(norm_ratings, weights=weights)
            predictions[anime] = round(user_mean + pred_deviation, 2)
    
    if not predictions:
        return {}
    recommendations = pd.Series(predictions).sort_values(ascending=False)
    return recommendations.head(n).to_dict()

In [8]:
t0 = time.time()

In [9]:
user_ratings = {
    #f"{entry['mal_id']}": f"{entry['score']}"
    f"{entry['mal_id']}_{entry['title']}": entry['score']
    for entry in user_data
    if entry.get('score') is not None and not (isinstance(entry['score'], float) and math.isnan(entry['score']))
}

print(user_ratings)

{'502_Dragon Ball Movie 1: Shen Long no Densetsu': 9, '891_Dragon Ball Movie 2: Majinjou no Nemurihime': 9, '892_Dragon Ball Movie 3: Makafushigi Daibouken': 9, '223_Dragon Ball': 10, '894_Dragon Ball Z Movie 01: Ora no Gohan wo Kaese!!': 10, '895_Dragon Ball Z Movie 02: Kono Yo de Ichiban Tsuyoi Yatsu': 10, '896_Dragon Ball Z Movie 03: Chikyuu Marugoto Choukessen': 10, '897_Dragon Ball Z Movie 04: Super Saiyajin da Son Gokuu': 10, '898_Dragon Ball Z Movie 05: Tobikkiri no Saikyou tai Saikyou': 10, '6714_Dragon Ball Z: Atsumare! Gokuu World': 10, '899_Dragon Ball Z Movie 06: Gekitotsu!! 100-oku Power no Senshi-tachi': 10, '900_Dragon Ball Z Movie 07: Kyokugen Battle!! Sandai Super Saiyajin': 10, '901_Dragon Ball Z Movie 08: Moetsukiro!! Nessen, Ressen, Chougekisen': 10, '902_Dragon Ball Z Movie 09: Ginga Girigiri!! Bucchigiri no Sugoi Yatsu': 10, '984_Dragon Ball Z: Saiya-jin Zetsumetsu Keikaku': 10, '903_Dragon Ball Z Movie 10: Kiken na Futari! Super Senshi wa Nemurenai': 10, '904_Dra

In [10]:
new_user_name = "Current_User"

new_user_row = pd.Series(0, index=df.columns, name=new_user_name)

for anime, rating in user_ratings.items():
    if anime in new_user_row.index:
        new_user_row[anime] = rating


df_extended = pd.concat([df, new_user_row.to_frame().T])
df_normalized = df_extended.apply(lambda row: row - row[row != 0].mean() if (row != 0).any() else row, axis=1)

In [11]:
n = 5
k = 10
max_amount_per_page = 25
delay = 1

In [12]:
recommendations = get_recommendations('Current_User', df_extended, df_normalized, n, k)

In [13]:
predicted = {}
if len(recommendations) < n:
    amount = (n - len(recommendations) + 1) // max_amount_per_page
    for page in range(1, amount + 2):
        url = f"https://api.jikan.moe/v4/top/anime"
        params = {
            "page": page,
        }
        
        response = requests.get(url, params=params)
        
        data = response.json();
        animes = data['data']

        for anime in data['data']:
            if len(recommendations) < n:
                recommendations[f"{str(anime['mal_id'])}_{anime['title']}"] = 0

In [14]:
for elem in recommendations:
    print(elem) 

51535_Shingeki no Kyojin: The Final Season - Kanketsu-hen
164_Mononoke Hime
9989_Ano Hi Mita Hana no Namae wo Bokutachi wa Mada Shiranai.
35851_Sayonara no Asa ni Yakusoku no Hana wo Kazarou
22135_Ping Pong the Animation


In [15]:
for anime_key, score in recommendations.items():
    id_ref, name = anime_key.split('_', 1)
    print(f"Recommend: {name} (ID: {id_ref}) with predicted score: {score}")

Recommend: Shingeki no Kyojin: The Final Season - Kanketsu-hen (ID: 51535) with predicted score: 11.65
Recommend: Mononoke Hime (ID: 164) with predicted score: 11.04
Recommend: Ano Hi Mita Hana no Namae wo Bokutachi wa Mada Shiranai. (ID: 9989) with predicted score: 11.04
Recommend: Sayonara no Asa ni Yakusoku no Hana wo Kazarou (ID: 35851) with predicted score: 11.04
Recommend: Ping Pong the Animation (ID: 22135) with predicted score: 11.04


In [16]:
print(time.time() - t0)

1.272991418838501
